<a href="https://colab.research.google.com/github/MaSBroEarl/Task.Text-generation/blob/main/TextGenerationLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Импортируем библиотеки:

In [ ]:
!pip install -q transformers torch accelerate

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from rouge_score import rouge_scorer
import nltk
import re
import time
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)


In [ ]:
# Проверяем GPU
print(f"CUDA доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Доступно памяти: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Загружаем данные:

In [ ]:
df = pd.read_csv('/content/train.csv')

In [ ]:
df.head()

Подготовка данных для обучения:

In [ ]:
#создаем выборку для инференса
# Берем 200 случайных диалогов
sample_size = 200
df_sample = df.sample(n=sample_size, random_state=42).reset_index(drop=True)
print(f"Используем {len(df_sample)} диалогов для инференса")

# Проверяем структуру
df_sample.head()

Загрузка модели

In [ ]:
# Выбираем модель
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

In [ ]:
# Загружаем токенизатор
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
# Загружаем модель
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

In [ ]:
# Проверяем модель на одном примере
def create_prompt(dialogue):
    prompt = f"""<|im_start|>system
You are a helpful assistant that summarizes dialogues concisely in 1-3 sentences.
<|im_end|>
<|im_start|>user
Summarize the following dialogue:

{dialogue}
<|im_end|>
<|im_start|>assistant
"""
    return prompt

# Берем первый диалог из выборки
test_dialogue = df_sample.iloc[0]['dialogue']
prompt = create_prompt(test_dialogue)

print("ТЕСТОВЫЙ ДИАЛОГ:")
print(test_dialogue[:200] + "...")

# Токенизируем
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)

# Переносим на GPU если есть
if torch.cuda.is_available():
    inputs = {k: v.to('cuda') for k, v in inputs.items()}

# Генерируем
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

# Декодируем
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Извлекаем суммаризацию
if "<|im_start|>assistant" in generated_text:
    summary = generated_text.split("<|im_start|>assistant")[-1].strip()
else:
    summary = generated_text[len(prompt):].strip()

print("\nСГЕНЕРИРОВАННАЯ СУММАРИЗАЦИЯ:")
print(summary)

print("\nРЕФЕРЕНСНАЯ СУММАРИЗАЦИЯ:")
print(df_sample.iloc[0]['summary'])

Запускаем генерацию для всех 200 примеров

In [ ]:
# Функция для генерации суммаризации
def generate_summary(dialogue, model, tokenizer, max_length=100):
    prompt = create_prompt(dialogue)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    if torch.cuda.is_available():
        inputs = {k: v.to('cuda') for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Извлекаем суммаризацию
    if "<|im_start|>assistant" in generated_text:
        summary = generated_text.split("<|im_start|>assistant")[-1].strip()
    else:
        summary = generated_text[len(prompt):].strip()

    return summary

print("Начинаем генерацию для 200 диалогов...")
start_time = time.time()

# Генерируем суммаризации для всех примеров
predictions = []
for i, row in df_sample.iterrows():
    dialogue = str(row['dialogue'])
    summary = generate_summary(dialogue, model, tokenizer)
    predictions.append(summary)

    # Показываем прогресс
    if (i + 1) % 20 == 0:
        print(f"Обработано {i+1}/{len(df_sample)} диалогов")

end_time = time.time()
print(f"Генерация завершена за {end_time - start_time:.2f} секунд")

# Добавляем предсказания в датафрейм
df_sample['predicted_summary'] = predictions
print("Предсказания сохранены!")

In [ ]:
# Функция для бейзлайна (первое + последнее предложение)
def first_last_summary(dialogue):
    if not isinstance(dialogue, str):
        return ""
    clean_dialogue = re.sub(r'#Person\d+#', '', dialogue)
    sentences = nltk.sent_tokenize(clean_dialogue)
    if len(sentences) == 0:
        return ""
    elif len(sentences) == 1:
        return sentences[0]
    else:
        first = sentences[0].strip()
        last = sentences[-1].strip()
        if first == last:
            return first
        return f"{first} {last}"

# Считаем бейзлайн для выборки
print("Считаем бейзлайн...")
df_sample['baseline_summary'] = df_sample['dialogue'].apply(first_last_summary)

# Инициализируем ROUGE scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Считаем метрики для Qwen
rouge1_qwen = []
rouge2_qwen = []
rougeL_qwen = []

# Считаем метрики для бейзлайна
rouge1_baseline = []
rouge2_baseline = []
rougeL_baseline = []

for idx, row in df_sample.iterrows():
    reference = str(row['summary'])
    pred_qwen = str(row['predicted_summary'])
    pred_baseline = str(row['baseline_summary'])

    # Qwen
    scores_qwen = scorer.score(reference, pred_qwen)
    rouge1_qwen.append(scores_qwen['rouge1'].fmeasure)
    rouge2_qwen.append(scores_qwen['rouge2'].fmeasure)
    rougeL_qwen.append(scores_qwen['rougeL'].fmeasure)

    # Бейзлайн
    scores_baseline = scorer.score(reference, pred_baseline)
    rouge1_baseline.append(scores_baseline['rouge1'].fmeasure)
    rouge2_baseline.append(scores_baseline['rouge2'].fmeasure)
    rougeL_baseline.append(scores_baseline['rougeL'].fmeasure)

# Средние значения
avg_qwen = {
    'rouge1': np.mean(rouge1_qwen),
    'rouge2': np.mean(rouge2_qwen),
    'rougeL': np.mean(rougeL_qwen)
}

avg_baseline = {
    'rouge1': np.mean(rouge1_baseline),
    'rouge2': np.mean(rouge2_baseline),
    'rougeL': np.mean(rougeL_baseline)
}

print("\n" + "="*50)
print("РЕЗУЛЬТАТЫ ОЦЕНКИ")
print("="*50)
print(f"\n{'Метрика':<15} {'Бейзлайн':<15} {'Qwen (zero-shot)':<20} {'Улучшение':<15}")
print("-"*65)
print(f"{'ROUGE-1':<15} {avg_baseline['rouge1']:.4f}       {avg_qwen['rouge1']:.4f}            {avg_qwen['rouge1'] - avg_baseline['rouge1']:.4f}")
print(f"{'ROUGE-2':<15} {avg_baseline['rouge2']:.4f}       {avg_qwen['rouge2']:.4f}            {avg_qwen['rouge2'] - avg_baseline['rouge2']:.4f}")
print(f"{'ROUGE-L':<15} {avg_baseline['rougeL']:.4f}       {avg_qwen['rougeL']:.4f}            {avg_qwen['rougeL'] - avg_baseline['rougeL']:.4f}")

# Относительное улучшение
improvement_pct = ((avg_qwen['rouge1'] - avg_baseline['rouge1']) / avg_baseline['rouge1']) * 100
print(f"\n📈 Относительное улучшение ROUGE-1: {improvement_pct:.1f}%")

In [ ]:
import matplotlib.pyplot as plt

# Собираем результаты в таблицу
results_df = pd.DataFrame({
    'Метрика': ['ROUGE-1', 'ROUGE-2', 'ROUGE-L'],
    'Бейзлайн': [avg_baseline['rouge1'], avg_baseline['rouge2'], avg_baseline['rougeL']],
    'Qwen (zero-shot)': [avg_qwen['rouge1'], avg_qwen['rouge2'], avg_qwen['rougeL']],
    'Улучшение': [
        avg_qwen['rouge1'] - avg_baseline['rouge1'],
        avg_qwen['rouge2'] - avg_baseline['rouge2'],
        avg_qwen['rougeL'] - avg_baseline['rougeL']
    ]
})

print("="*60)
print("ИТОГОВАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print("="*60)
print(results_df.to_string(index=False))

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# График 1: Сравнение метрик
x = np.arange(len(results_df['Метрика']))
width = 0.35

axes[0].bar(x - width/2, results_df['Бейзлайн'], width, label='Бейзлайн', color='skyblue')
axes[0].bar(x + width/2, results_df['Qwen (zero-shot)'], width, label='Qwen (zero-shot)', color='lightcoral')
axes[0].set_xlabel('Метрика')
axes[0].set_ylabel('Значение')
axes[0].set_title('Сравнение метрик ROUGE')
axes[0].set_xticks(x)
axes[0].set_xticklabels(results_df['Метрика'])
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# График 2: Улучшение
axes[1].bar(results_df['Метрика'], results_df['Улучшение'], color='green', alpha=0.7)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].set_xlabel('Метрика')
axes[1].set_ylabel('Абсолютное улучшение')
axes[1].set_title('Улучшение Qwen над бейзлайном')
axes[1].grid(True, alpha=0.3)

# Добавляем значения на столбцы
for i, v in enumerate(results_df['Улучшение']):
    axes[1].text(i, v + 0.001, f'{v:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('results_comparison.png', dpi=150)
plt.show()

print("\n✅ Графики сохранены в 'results_comparison.png'")